In [0]:
# 1. Importando Bibliotecas Necessárias
import requests
import pandas as pd
from datetime import datetime

In [0]:
# 2. Extraindo e Transformando Dados
def extrair_dados_bitcoin():
    """Extrai o JSON completo da API da Coinbase."""
    url = 'https://api.coinbase.com/v2/prices/spot'
    resposta = requests.get(url)
    return resposta.json()

In [0]:
extrair_dados_bitcoin()

{'data': {'amount': '69469.22', 'base': 'BTC', 'currency': 'USD'}}

In [0]:
def extrair_cotacao_usd_brl():
    """Extrai a cotação USD-BRL da API CurrencyFreaks."""
    api_key = 'bf8ddcbf744d4e7897e836941c32e39f'
    url = f'https://api.currencyfreaks.com/v2.0/rates/latest?apikey={api_key}'
    resposta = requests.get(url)
    return resposta.json()

In [0]:
extrair_cotacao_usd_brl()

{'date': '2026-03-26 00:00:00+00',
 'base': 'USD',
 'rates': {'AGLD': '4.076451257038973',
  'FJD': '2.24266',
  'SCR': '14.7808',
  'BBD': '2.0',
  'HNL': '26.5366',
  'UGX': '3723.37',
  'COSMOSDYDX': '11.347194251352487',
  'NEAR': '0.7818608287724785',
  'AIOZ': '15.723270440251572',
  'AUDIO': '15.985656683336428',
  'WLD': '3.086533681112919',
  'HNT': '0.8032128514056225',
  'ETHFI': '1.7880822336741065',
  'FARM': '0.0832672225182845',
  'A8': '100.27599432711924',
  'SDG': '600.171',
  'DGB': '232.2669905964887',
  'AB': '493.73096439199423',
  'BCH': '0.0021119547196908',
  'AI': '48.42766379809753',
  'FKP': '0.748307',
  'JST': '16.79224141965687',
  'HOT': '2244.530502751359',
  'AO': '0.32978746190127795',
  'AR': '0.5369599684081697',
  'B3': '3086.41975308642',
  'CHILLGUY': '126.4337631839047',
  'SEI': '1.0',
  'QAI': '0.015837922745034965',
  'SEK': '9.35352',
  'LION': '354.69470962554107',
  'BB': '39.9067347777325',
  'QAR': '3.6455',
  'JTO': '2.9931158335827597'

In [0]:
def tratar_dados_bitcoin(dados_json, taxa_usd_brl):
    """Transforma os dados brutos da API, renomeia colunas, adiciona timestamp e converte para BRL."""
    valor_usd = float(dados_json['data']['amount'])
    criptomoeda = dados_json['data']['base']
    moeda_original = dados_json['data']['currency']
    
    # Convertendo de USD para BRL
    valor_brl = valor_usd * taxa_usd_brl
    
    # Adicionando timestamp como datetime object
    timestamp = datetime.now()
    
    dados_tratados = [{
        "valor_usd": valor_usd,
        "valor_brl": valor_brl,
        "criptomoeda": criptomoeda,
        "moeda_original": moeda_original,
        "taxa_conversao_usd_brl": taxa_usd_brl,
        "timestamp": timestamp
    }]
    
    return dados_tratados

In [0]:
# Extraindo dados
Dados_bitcoin = extrair_dados_bitcoin()
dados_cotacao = extrair_cotacao_usd_brl()

In [0]:
# Extraindo a taxa de conversão USD-BRL
taxa_usd_brl = float(dados_cotacao['rates']['BRL'])

In [0]:
# Tratando os dados e convertendo para BRL
dados_bitcoin_tratado = tratar_dados_bitcoin(Dados_bitcoin, taxa_usd_brl)


In [0]:
%sql
CREATE CATALOG IF NOT EXISTS pipeline_api_bitcoin
COMMENT 'Catálogo de demonstração criado para o workshop de pipeline_api_bitcoin';


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS pipeline_api_bitcoin.lakehouse
COMMENT 'Schema Lakehouse para salvar dados processados';


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS pipeline_api_bitcoin.lakehouse.raw_files
COMMENT 'Volume para arquivos brutos de ingestão inicial';


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS pipeline_api_bitcoin.bitcoin_data
COMMENT 'Schema para armazenar dados de Bitcoin processados';

In [0]:
## 4. Criando DataFrame Pandas

# Criar DataFrame Pandas a partir dos dados tratados
df_pandas = pd.DataFrame(dados_bitcoin_tratado)

In [0]:
df_pandas

,valor_usd,valor_brl,criptomoeda,moeda_original,taxa_conversao_usd_brl,timestamp
0,69469.22,363060.037564,BTC,USD,5.2262,2026-03-26 14:24:57.915988


In [0]:
# 5. Salvando em JSON usando Pandas

# Pega o timestamp do próprio evento
event_ts = dados_bitcoin_tratado[0]["timestamp"]

# Converte para formato seguro para nome de arquivo
ts = event_ts.strftime("%Y%m%d_%H%M%S_%f")

# Caminho do arquivo JSON
json_file = f"/Volumes/pipeline_api_bitcoin/lakehouse/raw_files/bitcoin_{ts}.json"

# Salvar como JSON usando Pandas
df_pandas.to_json(json_file, orient='records', date_format='iso', indent=2)
print(f"✅ Arquivo JSON salvo: {json_file}")


✅ Arquivo JSON salvo: /Volumes/pipeline_api_bitcoin/lakehouse/raw_files/bitcoin_20260326_142457_915988.json


In [0]:
# Caminho do arquivo CSV
csv_file = f"/Volumes/pipeline_api_bitcoin/lakehouse/raw_files/bitcoin_{ts}.csv"

# Salvar como CSV usando Pandas
df_pandas.to_csv(csv_file, index=False)
print(f"✅ Arquivo CSV salvo: {csv_file}")

✅ Arquivo CSV salvo: /Volumes/pipeline_api_bitcoin/lakehouse/raw_files/bitcoin_20260326_142457_915988.csv


In [0]:
# Caminho do arquivo Parquet
parquet_file = f"/Volumes/pipeline_api_bitcoin/lakehouse/raw_files/bitcoin_{ts}.parquet"

# Salvar como Parquet usando Pandas
df_pandas.to_parquet(parquet_file, index=False)
print(f"✅ Arquivo Parquet salvo: {parquet_file}")

✅ Arquivo Parquet salvo: /Volumes/pipeline_api_bitcoin/lakehouse/raw_files/bitcoin_20260326_142457_915988.parquet


In [0]:
# 8. Convertendo Pandas para PySpark e Salvando como Delta Table

df_spark = spark.createDataFrame(df_pandas)

In [0]:
# Mostrar schema
df_spark.printSchema()

root
 |-- valor_usd: double (nullable = true)
 |-- valor_brl: double (nullable = true)
 |-- criptomoeda: string (nullable = true)
 |-- moeda_original: string (nullable = true)
 |-- taxa_conversao_usd_brl: double (nullable = true)
 |-- timestamp: timestamp (nullable = true)



In [0]:
# 9. Salvando como Delta Table
# Caminho da tabela Delta no Unity Catalog (schema bitcoin_data)
delta_table_path = "pipeline_api_bitcoin.bitcoin_data.bitcoin_data"


In [0]:
# Salvar como Delta Table usando o DataFrame PySpark (modo append se a tabela já existir)
df_spark.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(delta_table_path)

In [0]:
%sql
SELECT * FROM pipeline_api_bitcoin.bitcoin_data.bitcoin_data

valor_usd,valor_brl,criptomoeda,moeda_original,taxa_conversao_usd_brl,timestamp
69550.64,363485.55476800003,BTC,USD,5.2262,2026-03-26T14:14:22.752Z
69470.575,363067.11906500004,BTC,USD,5.2262,2026-03-26T14:23:14.397Z
69628.105,363890.402351,BTC,USD,5.2262,2026-03-26T14:16:11.778Z
69610.505,363798.42123100004,BTC,USD,5.2262,2026-03-26T11:04:39.363Z
69361.215,362495.581833,BTC,USD,5.2262,2026-03-26T13:27:50.323Z
69573.015,363602.49099300004,BTC,USD,5.2262,2026-03-26T14:15:37.634Z
70705.495,370037.2080825,BTC,USD,5.2335,2026-03-25T01:18:10.249Z
69469.22,363060.03756400006,BTC,USD,5.2262,2026-03-26T14:24:57.915Z
69672.225,364120.98229500005,BTC,USD,5.2262,2026-03-26T13:55:46.539Z
69417.045,362787.36057900003,BTC,USD,5.2262,2026-03-26T14:22:39.829Z


In [0]:
# 10. Convertendo Delta Table para DataFrame
df_delta = spark.read.table(delta_table_path)

In [0]:
df_delta.printSchema()

root
 |-- valor_usd: double (nullable = true)
 |-- valor_brl: double (nullable = true)
 |-- criptomoeda: string (nullable = true)
 |-- moeda_original: string (nullable = true)
 |-- taxa_conversao_usd_brl: double (nullable = true)
 |-- timestamp: timestamp (nullable = true)



In [0]:
df_delta.show(truncate=False)

+---------+------------------+-----------+--------------+----------------------+--------------------------+
|valor_usd|valor_brl         |criptomoeda|moeda_original|taxa_conversao_usd_brl|timestamp                 |
+---------+------------------+-----------+--------------+----------------------+--------------------------+
|69550.64 |363485.55476800003|BTC        |USD           |5.2262                |2026-03-26 14:14:22.75249 |
|69470.575|363067.11906500004|BTC        |USD           |5.2262                |2026-03-26 14:23:14.397116|
|69628.105|363890.402351     |BTC        |USD           |5.2262                |2026-03-26 14:16:11.778583|
|69610.505|363798.42123100004|BTC        |USD           |5.2262                |2026-03-26 11:04:39.363024|
|69361.215|362495.581833     |BTC        |USD           |5.2262                |2026-03-26 13:27:50.323731|
|69573.015|363602.49099300004|BTC        |USD           |5.2262                |2026-03-26 14:15:37.634179|
|70705.495|370037.2080825   

In [0]:
%sql
select * from pipeline_api_bitcoin.bitcoin_data.bitcoin_data
order by timestamp desc
limit 10;

valor_usd,valor_brl,criptomoeda,moeda_original,taxa_conversao_usd_brl,timestamp
69469.22,363060.03756400006,BTC,USD,5.2262,2026-03-26T14:24:57.915Z
69470.575,363067.11906500004,BTC,USD,5.2262,2026-03-26T14:23:14.397Z
69417.045,362787.36057900003,BTC,USD,5.2262,2026-03-26T14:22:39.829Z
69628.105,363890.402351,BTC,USD,5.2262,2026-03-26T14:16:11.778Z
69573.015,363602.49099300004,BTC,USD,5.2262,2026-03-26T14:15:37.634Z
69550.64,363485.55476800003,BTC,USD,5.2262,2026-03-26T14:14:22.752Z
69672.225,364120.98229500005,BTC,USD,5.2262,2026-03-26T13:55:46.539Z
69361.215,362495.581833,BTC,USD,5.2262,2026-03-26T13:27:50.323Z
69610.505,363798.42123100004,BTC,USD,5.2262,2026-03-26T11:04:39.363Z
70705.495,370037.2080825,BTC,USD,5.2335,2026-03-25T01:18:10.249Z


In [0]:
%sql
DESCRIBE HISTORY pipeline_api_bitcoin.bitcoin_data.bitcoin_data

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
9,2026-03-26T14:25:05.000Z,78885150469514,thalysstark715@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(3952493653254192),8dfef95a-69a9-40d5-abc5-d048a6090e4c,0326-135547-xzp2429y-v2n,8,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1780)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
8,2026-03-26T14:23:22.000Z,78885150469514,thalysstark715@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(3952493653254192),86694fc2-02f9-45cc-a007-e12084c26ec4,0326-135547-xzp2429y-v2n,7,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1781)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
7,2026-03-26T14:22:47.000Z,78885150469514,thalysstark715@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(3952493653254192),520425d3-a819-48d8-ae5d-46ff3dc48e2b,0326-135547-xzp2429y-v2n,6,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1780)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
6,2026-03-26T14:16:19.000Z,78885150469514,thalysstark715@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(3952493653254192),ae5a658e-2d10-47b9-b498-a792cf2451d8,0326-135547-xzp2429y-v2n,5,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1781)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
5,2026-03-26T14:15:46.000Z,78885150469514,thalysstark715@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(3952493653254192),d98644d6-37ab-4704-958b-864320479d21,0326-135547-xzp2429y-v2n,4,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1781)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
4,2026-03-26T14:14:41.000Z,78885150469514,thalysstark715@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(3952493653254192),56f28c7f-8f10-43ae-bf7f-5c985459beb2,0326-135547-xzp2429y-v2n,3,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1781)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
3,2026-03-26T13:56:42.000Z,78885150469514,thalysstark715@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(3952493653254192),23360903-8e04-4f20-a0a5-d43948445c2b,0326-135547-xzp2429y-v2n,2,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1780)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
2,2026-03-26T13:28:08.000Z,78885150469514,thalysstark715@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(3952493653254192),3e16f4e9-92c4-444e-bc72-18215d5552ec,0326-131352-20305vgi-v2n,1,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1781)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
1,2026-03-26T11:05:35.000Z,78885150469514,thalysstark715@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(3952493653254192),7a59d41e-dd25-4140-9181-8831b789a88f,0326-110440-pfjk0q8z-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1781)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
0,2026-03-25T01:24:08.000Z,78885150469514,thalysstark715@gmail.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3952493653254192),9864eed7-9d6e-4658-9091-f6c480db2a25,0325-002145-jylch8sz-v2n,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes